# Increments 4–5 — Gate quality and deployment-stream simulation
Attach outputs of 01–03. Runs Table 4.7 and the conditions of Table 3.6. Each condition takes roughly 1–2 h on T4 for ~1,500 stream cases: run a few conditions per session and Save Version; later sessions attach this notebook's output and continue.

In [ ]:
# === STEP 0: SETUP — run this cell first, every time =========================
# Before running: in the right-hand panel
#   * "Session options" -> Accelerator = GPU T4 x2, Internet = On
#   * "Add Input" -> your code dataset (the uploaded bcd-project.zip)
#   * "Add Input" -> search "brain tumor mri dataset" by masoudnickparvar
#   * (notebooks 02-05) "Add Input" -> "Your Work" -> the previous notebook(s)
import os, sys, glob, shutil, subprocess

def find_dir(test, root='/kaggle/input', max_depth=6):
    # walk the attached inputs and return the first folder that passes test(folder)
    root_depth = root.rstrip('/').count('/')
    for d, subdirs, files in os.walk(root):
        if d.count('/') - root_depth > max_depth:
            subdirs[:] = []
            continue
        if test(d, subdirs, files):
            return d
    return None

# 1) the project code: the folder that contains bcd/__init__.py
code_dir = find_dir(lambda d, s, f: os.path.isfile(os.path.join(d, 'bcd', '__init__.py')))
if code_dir is None:
    raise SystemExit('Code not found. Click "Add Input" and attach the dataset you created from bcd-project.zip.')
shutil.copytree(code_dir, '/kaggle/working/bcd-project', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/bcd-project')
print('code found at:', code_dir)

# 2) install the extra libraries (takes ~1-2 minutes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<4.52', 'peft>=0.12',
                'bitsandbytes>=0.43', 'accelerate>=0.33', 'ultralytics', 'imagehash', 'h5py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])  # old torchao on Kaggle breaks peft

# 3) the brain MRI dataset: the folder that has Training/ and Testing/ inside
from bcd.config import load_config
cfg = load_config('/kaggle/working/bcd-project/configs/default.yaml')
mri = find_dir(lambda d, s, f: 'Training' in s and 'Testing' in s)
if mri is None:
    raise SystemExit('MRI dataset not found. Click "Add Input" and attach masoudnickparvar/brain-tumor-mri-dataset.')
cfg.paths.kaggle_root = mri
print('MRI dataset found at:', mri)

# 4) results of earlier notebooks (a folder named bcd that contains manifest.csv)
os.makedirs(cfg.paths.work_dir, exist_ok=True)
for prev in glob.glob('/kaggle/input/**/bcd/manifest.csv', recursive=True):
    src = os.path.dirname(prev)
    if not src.startswith('/kaggle/working'):
        shutil.copytree(src, cfg.paths.work_dir, dirs_exist_ok=True)
        print('restored earlier results from:', src)
print('GPUs available:', __import__('torch').cuda.device_count())
print('Setup complete. Work folder:', cfg.paths.work_dir)


In [ ]:
import glob, json
from bcd.runtime import build_runtime, run_condition, gate_quality
W = cfg.paths.work_dir
yolo = glob.glob(f'{W}/yolo_runs/tumour/weights/best.pt')[0]
rt = build_runtime(cfg, f'{W}/adapters/student_seed', f'{W}/adapters/teacher', yolo)
print(len(rt['stream']), 'stream cases;', 'teacher T =', rt['teacher'].T)
from bcd.runtime import teacher_test_eval
print('teacher on test:', teacher_test_eval(cfg, rt))

In [ ]:
import pandas as pd
if os.path.exists(f'{W}/table_gate_quality.csv') and 'gate_quality' not in globals().get('FORCE', []):
    gq = pd.read_csv(f'{W}/table_gate_quality.csv')
else:
    gq = gate_quality(cfg, rt); gq.to_csv(f'{W}/table_gate_quality.csv', index=False)
print(gq.to_string())   # Table 4.10

In [ ]:
# Choose what to run in this session. P must run before B2, B3 and B5 (they use P's escalation rate).
import shutil
for cond in globals().get('FORCE', []):   # re-run these: the old run is moved to runs_archive (kept for the report)
    if os.path.isdir(f'{W}/runs/{cond}'):
        os.makedirs(f'{W}/runs_archive', exist_ok=True)
        shutil.rmtree(f'{W}/runs_archive/{cond}', ignore_errors=True)
        shutil.move(f'{W}/runs/{cond}', f'{W}/runs_archive/{cond}'); print('archived old run', cond)
TODO = globals().get('RUN_LIST') or [('P', 0), ('B0', 0), ('B1', 0), ('B2', 0)]   # P first: B2/B3/B5 use its escalation rate
import traceback
FAILED = []
for cond, seed in TODO:
    if os.path.exists(f'{W}/runs/{cond}_seed{seed}/summary.json'):
        print('skip', cond, seed); continue
    try:
        rr = None
        if cond.split('_')[0] in ('B2', 'B3', 'B5'):   # matched escalation budget = the proposed system's rate (P, or P_v2 for _v2 runs)
            ref = 'P_v2' if cond.endswith('_v2') else 'P'
            rr = json.load(open(f'{W}/runs/{ref}_seed{seed}/summary.json'))['escalation_rate']
        s = run_condition(cfg, rt, cond, seed, random_rate=rr)
        print(cond, seed, {k: s[k] for k in ('escalation_rate', 'reviews_per_100')}, s['final_student_test'])
    except Exception:
        traceback.print_exc(); FAILED.append(cond)   # keep going: the other runs still finish and are saved
        print('RUN FAILED:', cond, '- the other runs continue')
print('failed runs:', FAILED or 'none')
from bcd.runtime import results_dump
results_dump(cfg)

Suggested sessions (seed 0): `P B0 B1 B2` → `B3 B4 B5` → `A1 A2` → `A3 A4`; extra seeds for P, B3, B5 only if GPU hours remain.